# 19 — Target Model Review and Map-Ready Outputs

This notebook reviews the first scoring output and prepares map-ready / reporting-ready files.

It does **not** change the scoring model. Its role is quality assurance, interpretation, and export.

Default scope: **North West**

## 19.1 Project paths and switches

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

MODEL_DIR = PROCESSED_DIR / "target_model_v1"
MODEL_INPUT_DIR = MODEL_DIR / "inputs"
MODEL_OUTPUT_DIR = MODEL_DIR / "outputs"
MODEL_REVIEW_DIR = MODEL_DIR / "review"

REVIEW_OUTPUT_DIR = MODEL_REVIEW_DIR / "model_review_outputs_v1"
MAP_READY_DIR = REVIEW_OUTPUT_DIR / "map_ready"
TABLE_DIR = REVIEW_OUTPUT_DIR / "tables"

for d in [REVIEW_OUTPUT_DIR, MAP_READY_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEFAULT_SCOPE_NAME = "north_west"
DEFAULT_SCOPE_DIR = MODEL_OUTPUT_DIR / "scopes" / DEFAULT_SCOPE_NAME

print("Project:", PROJECT_DIR)
print("Review output:", REVIEW_OUTPUT_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Review output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\review\model_review_outputs_v1


## 19.2 Load scored files

The notebook first loads the default North West scoped file. It also loads the all-available scored file for national context.

In [2]:
NW_SCORE_PATH = DEFAULT_SCOPE_DIR / f"initial_watchlist_scores_{DEFAULT_SCOPE_NAME}_v1.csv"
ALL_SCORE_PATH = MODEL_OUTPUT_DIR / "initial_watchlist_scores_ward25_all_available_v1.csv"

if not NW_SCORE_PATH.exists():
    raise FileNotFoundError(f"Missing North West score output: {NW_SCORE_PATH}. Run Notebook 18 first.")

nw = pd.read_csv(NW_SCORE_PATH, low_memory=False)

if ALL_SCORE_PATH.exists():
    all_scores = pd.read_csv(ALL_SCORE_PATH, low_memory=False)
else:
    all_scores = pd.DataFrame()

print("North West rows:", len(nw))
print("All available rows:", len(all_scores))
display(nw.head())

North West rows: 825
All available rows: 7572


,LAD25CD,LAD25NM,WD25CD,WD25NM,RGN25CD,RGN25NM,analysis_region,country_inferred,scope_north_west,scope_england,scope_wales,scope_all_available,population,oa_count,dominant_cluster,dominant_cluster_name,second_cluster,second_cluster_name,dominant_cluster_share,second_cluster_share,cluster_fragmentation_index,is_mixed_ward,is_clear_dominant_ward,is_highly_fragmented,cluster_0_share,cluster_1_share,cluster_2_share,cluster_3_share,cluster_4_share,cluster_5_share,cluster_6_share,student_transient_youth_share,rooted_older_homeowners_share,stable_suburban_professionals_share,cosmopolitan_young_professional_core_share,settled_working_families_skilled_trades_suburbs_share,settled_diverse_urban_communities_share,post_industrial_estates_deprived_working_communities_share,age_0_14_pct,age_15_24_pct,age_25_34_pct,age_35_49_pct,age_50_64_pct,age_65_plus_pct,uk_born_pct,non_uk_born_pct,resident_10_plus_years_pct,resident_less_5_years_pct,white_british_pct,white_other_pct,non_white_pct,owned_pct,owns_outright_pct,owns_mortgage_pct,social_rented_pct,private_rented_pct,house_type_pct,flat_type_pct,managerial_professional_pct,skilled_traditional_pct,routine_service_elementary_pct,employed_pct,unemployed_pct,full_time_student_pct,retired_pct,long_term_sick_disabled_pct,no_qualifications_pct,level_1_2_pct,apprenticeship_pct,level_4_plus_pct,one_person_household_pct,married_couple_family_pct,lone_parent_family_pct,latest_election_WD25NM,latest_election_LAD25CD,latest_election_LAD25NM,latest_election_source_year,latest_election_allocated_electorate,latest_election_allocated_valid_votes,latest_election_allocated_ballots,latest_election_allocated_invalid_votes,latest_election_allocated_top_party_votes,latest_election_allocated_runner_up_party_votes,latest_election_allocated_con_votes,latest_election_allocated_lab_votes,latest_election_allocated_ld_votes,latest_election_allocated_green_votes,latest_election_allocated_reform_ukip_brexit_votes,latest_election_allocated_independent_votes,latest_election_allocated_sdp_votes,latest_election_allocated_other_votes,latest_election_contributing_oa_rows,latest_election_contributing_result_areas,latest_election_contributing_source_years,latest_election_con_share,latest_election_lab_share,latest_election_ld_share,latest_election_green_share,latest_election_reform_ukip_brexit_share,latest_election_independent_share,latest_election_sdp_share,latest_election_other_share,latest_election_top_party_bucket,latest_election_runner_up_party_bucket,latest_election_top_party_votes_allocated,latest_election_runner_up_party_votes_allocated,latest_election_margin_votes_allocated,latest_election_margin_pct_allocated,latest_election_party_fragmentation_index,latest_election_effective_number_of_parties,latest_election_aggregation_label,latest_election_latest_layer_note,has_latest_election_layer,has_valid_vote_data,has_margin_data,boundary_caveat,county_election_caveat,target_model_ready,data_confidence_note,demographic_relevance_raw,demographic_relevance_score,margin_competitiveness_score,top_party_dominance_inverse_score,valid_vote_threshold_score,electoral_fragmentation_score,electoral_opportunity_score,non_main_party_share,lab_con_combined_share,lab_con_inverse_score,non_main_party_score,effective_parties_score,political_openness_score,data_confidence_score,initial_watchlist_score,initial_watchlist_percentile,review_band
0,E07000117,Burnley,E05005152,Brunshaw,E12000002,North West,North West,England,True,True,False,True,6264,23,6.0,Post-Industrial Estates / Deprived Working Com...,4.0,Settled Working Families / Skilled Trades Suburbs,0.633780,0.261494,0.518976,False,True,False,0.0,0.104725,0.000000,0.0,0.261494,0.000000,0.633780,0.0,0.104725,0.000000,0.0,0.261494,0.000000,0.633780,0.172254,0.104566,0.122765,0.188059,0.213921,0.197957,0.952693,0.047307,0.028439,0.011184,0.936782,0.019955,0.035920,0.525856,0.280725,0.245131,0.291471,0.178643,0.830868,0.168124,0.289866,0.221311,0.397914,0.524824,0.034793,0.060594,0.225567,0.093628,0

## 19.3 Coverage and caveat review

In [3]:
coverage_cols = [
    "target_model_ready",
    "has_latest_election_layer",
    "has_valid_vote_data",
    "has_margin_data",
    "data_confidence_note",
    "boundary_caveat",
    "county_election_caveat",
]

coverage_cols = [c for c in coverage_cols if c in nw.columns]

coverage_summary = {}

for col in coverage_cols:
    coverage_summary[col] = nw[col].value_counts(dropna=False).to_dict()

coverage_summary_df = pd.DataFrame([
    {"field": field, "value": key, "rows": value}
    for field, counts in coverage_summary.items()
    for key, value in counts.items()
])

coverage_summary_df.to_csv(TABLE_DIR / "north_west_model_coverage_and_caveats_v1.csv", index=False)
display(coverage_summary_df)

,field,value,rows
0,target_model_ready,True,824
1,target_model_ready,False,1
2,has_latest_election_layer,True,825
3,has_valid_vote_data,True,824
4,has_valid_vote_data,False,1
5,has_margin_data,True,824
6,has_margin_data,False,1
7,data_confidence_note,No major caveat.,671
8,data_confidence_note,County election caveat.,131
9,data_confidence_note,Boundary caveat.,22


## 19.4 Score distribution and component review

This section helps identify whether the combined score is being driven mainly by one component.

In [4]:
score_cols = [
    "initial_watchlist_score",
    "demographic_relevance_score",
    "electoral_opportunity_score",
    "political_openness_score",
    "data_confidence_score",
]

score_cols = [c for c in score_cols if c in nw.columns]

score_distribution = nw[score_cols].describe().T.reset_index().rename(columns={"index": "score_component"})
score_distribution.to_csv(TABLE_DIR / "north_west_score_distribution_v1.csv", index=False)
display(score_distribution)

score_corr = nw[score_cols].corr()
score_corr.to_csv(TABLE_DIR / "north_west_score_component_correlation_v1.csv")
display(score_corr)

,score_component,count,mean,std,min,25%,50%,75%,max
0,initial_watchlist_score,825.0,43.407355,7.631335,22.219767,37.950182,43.948770,48.989457,61.603515
1,demographic_relevance_score,825.0,22.274619,11.393042,0.000000,13.537886,23.041015,31.448778,45.000000
2,electoral_opportunity_score,825.0,57.971570,16.600304,8.881188,46.392217,61.778999,71.295651,88.394661
3,political_openness_score,825.0,34.115433,14.754747,0.000000,21.983567,36.391432,45.946086,61.202571
4,data_confidence_score,825.0,96.909091,4.879393,50.000000,90.000000,100.000000,100.000000,100.000000


,initial_watchlist_score,demographic_relevance_score,electoral_opportunity_score,political_openness_score,data_confidence_score
initial_watchlist_score,1.000000,0.283249,0.820643,0.684889,-0.228225
demographic_relevance_score,0.283249,1.000000,-0.161912,-0.266099,-0.078084
electoral_opportunity_score,0.820643,-0.161912,1.000000,0.547776,-0.189420
political_openness_score,0.684889,-0.266099,0.547776,1.000000,-0.264296
data_confidence_score,-0.228225,-0.078084,-0.189420,-0.264296,1.000000


## 19.5 Council-level review table

In [5]:
council_review = (
    nw.groupby(["LAD25CD", "LAD25NM"], as_index=False)
    .agg(
        wards=("WD25CD", "nunique"),
        population=("population", "sum") if "population" in nw.columns else ("WD25CD", "size"),
        mean_initial_watchlist_score=("initial_watchlist_score", "mean"),
        median_initial_watchlist_score=("initial_watchlist_score", "median"),
        max_initial_watchlist_score=("initial_watchlist_score", "max"),
        review_a_count=("review_band", lambda s: (s == "Review A").sum()),
        review_b_count=("review_band", lambda s: (s == "Review B").sum()),
        ready_wards=("target_model_ready", "sum") if "target_model_ready" in nw.columns else ("WD25CD", "size"),
        mean_demographic_relevance_score=("demographic_relevance_score", "mean"),
        mean_electoral_opportunity_score=("electoral_opportunity_score", "mean"),
        mean_political_openness_score=("political_openness_score", "mean"),
        mean_data_confidence_score=("data_confidence_score", "mean"),
    )
    .sort_values(["mean_initial_watchlist_score"], ascending=False)
)

council_review["ready_share"] = council_review["ready_wards"] / council_review["wards"]

council_review.to_csv(TABLE_DIR / "north_west_council_review_summary_v1.csv", index=False)
display(council_review.head(30))

,LAD25CD,LAD25NM,wards,population,mean_initial_watchlist_score,median_initial_watchlist_score,max_initial_watchlist_score,review_a_count,review_b_count,ready_wards,mean_demographic_relevance_score,mean_electoral_opportunity_score,mean_political_openness_score,mean_data_confidence_score,ready_share
16,E07000125,Rossendale,10,70870,54.343783,55.697335,60.815359,5,2,10,24.490547,76.734707,52.206716,97.000000,1.000000
1,E06000007,Warrington,22,210978,51.332618,51.920404,59.964809,5,6,22,19.602942,74.286599,48.742434,100.000000,1.000000
10,E07000119,Fylde,17,81383,50.681101,52.083192,58.660754,1,9,17,20.601039,76.245423,44.741385,94.117647,1.000000
8,E07000117,Burnley,15,94653,50.443367,50.576298,61.603515,2,5,15,26.215176,65.525621,49.374809,92.666667,1.000000
12,E07000121,Lancaster,27,142923,49.006427,48.281585,61.230889,6,4,27,19.984542,66.877653,50.460834,93.333333,1.000000
17,E07000126,South Ribble,23,111021,48.210369,50.108867,54.728155,0,7,23,23.180277,69.145722,40.544658,92.173913,1.000000
20,E08000001,Bolton,20,295971,48.099147,49.339265,57.888177,1,4,20,21.197741,67.962657,41.164561,100.000000,1.000000
23,E08000004,Oldham,20,242115,47.475661,48.256531,58.064163,3,3,20,21.655253,62.443998,44.652494,100.000000,1.000000
11,E07000120,Hyndburn,16,82237,47.382527,49.634064,52.809123,0,3,16,27.371541,65.948004,34.572346,93.750000,1.000000
15,E07000124,Ribble Valley,26,61558,47.271394,48.462036,58.350736,4,4,25,12.630058,72.540022,48.355467,90.000000,0.961538


## 19.6 Review-band composition by tribe and top party

These tables are useful for checking whether the model is behaving coherently.

In [6]:
if "dominant_cluster_name" in nw.columns:
    tribe_band = pd.crosstab(
        nw["dominant_cluster_name"],
        nw["review_band"],
        normalize="index"
    ).reset_index()

    tribe_band.to_csv(TABLE_DIR / "north_west_review_band_by_dominant_cluster_v1.csv", index=False)
    display(tribe_band)

if "latest_election_top_party_bucket" in nw.columns:
    party_band = pd.crosstab(
        nw["latest_election_top_party_bucket"],
        nw["review_band"],
        normalize="index"
    ).reset_index()

    party_band.to_csv(TABLE_DIR / "north_west_review_band_by_latest_top_party_v1.csv", index=False)
    display(party_band)

review_band,dominant_cluster_name,Manual / incomplete data,Review A,Review B,Review C,Review D
0,Cosmopolitan Young Professional Core,0.000000,0.000000,0.000000,0.111111,0.888889
1,Post-Industrial Estates / Deprived Working Com...,0.000000,0.078313,0.138554,0.216867,0.566265
2,Rooted Older Homeowners,0.000000,0.038627,0.115880,0.356223,0.489270
3,Settled Diverse Urban Communities,0.000000,0.000000,0.017241,0.155172,0.827586
4,Settled Working Families / Skilled Trades Suburbs,0.000000,0.084337,0.156627,0.331325,0.427711
5,Stable Suburban Professionals,0.005848,0.000000,0.023392,0.175439,0.795322
6,Student & Transient Youth,0.000000,0.000000,0.000000,0.045455,0.954545


review_band,latest_election_top_party_bucket,Manual / incomplete data,Review A,Review B,Review C,Review D
0,con,0.008696,0.008696,0.060870,0.217391,0.704348
1,green,0.000000,0.000000,0.029412,0.441176,0.529412
2,independent,0.000000,0.105263,0.175439,0.263158,0.456140
3,lab,0.000000,0.034568,0.049383,0.182716,0.733333
4,ld,0.000000,0.068182,0.125000,0.215909,0.590909
5,other,0.000000,0.035714,0.107143,0.464286,0.392857
6,reform_ukip_brexit,0.000000,0.081633,0.295918,0.551020,0.071429


## 19.7 Map-ready score files

These files are designed to be joined to Ward25 boundaries in QGIS/GeoPandas using `WD25CD`.

In [7]:
map_cols = [
    "LAD25CD", "LAD25NM", "WD25CD", "WD25NM",
    "analysis_region",
    "population",
    "dominant_cluster_name",
    "second_cluster_name",
    "dominant_cluster_share",
    "cluster_fragmentation_index",
    "latest_election_source_year",
    "latest_election_top_party_bucket",
    "latest_election_runner_up_party_bucket",
    "latest_election_margin_pct_allocated",
    "latest_election_party_fragmentation_index",
    "initial_watchlist_score",
    "initial_watchlist_percentile",
    "review_band",
    "demographic_relevance_score",
    "electoral_opportunity_score",
    "political_openness_score",
    "data_confidence_score",
    "target_model_ready",
    "data_confidence_note",
    "boundary_caveat",
    "county_election_caveat",
]

map_cols = [c for c in map_cols if c in nw.columns]

nw_map = nw[map_cols].copy()

# Safer field names for GIS exports if needed.
nw_map.to_csv(MAP_READY_DIR / "north_west_ward25_initial_score_map_ready_v1.csv", index=False)

# Separate map-ready layers for each component.
for component in [
    "initial_watchlist_score",
    "demographic_relevance_score",
    "electoral_opportunity_score",
    "political_openness_score",
    "data_confidence_score",
]:
    if component in nw.columns:
        comp_map = nw[["LAD25CD", "LAD25NM", "WD25CD", "WD25NM", component, "review_band"]].copy()
        comp_map.to_csv(MAP_READY_DIR / f"north_west_ward25_{component}_map_ready_v1.csv", index=False)

print("Saved map-ready files:", MAP_READY_DIR)
display(nw_map.head())

Saved map-ready files: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\review\model_review_outputs_v1\map_ready


,LAD25CD,LAD25NM,WD25CD,WD25NM,analysis_region,population,dominant_cluster_name,second_cluster_name,dominant_cluster_share,cluster_fragmentation_index,latest_election_source_year,latest_election_top_party_bucket,latest_election_runner_up_party_bucket,latest_election_margin_pct_allocated,latest_election_party_fragmentation_index,initial_watchlist_score,initial_watchlist_percentile,review_band,demographic_relevance_score,electoral_opportunity_score,political_openness_score,data_confidence_score,target_model_ready,data_confidence_note,boundary_caveat,county_election_caveat
0,E07000117,Burnley,E05005152,Brunshaw,North West,6264,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,0.633780,0.518976,2025.0,independent,reform_ukip_brexit,0.074756,0.734334,61.603515,99.550977,Review A,39.766922,77.948165,61.202571,90.0,True,County election caveat.,NaN,Latest election layer may include 2025 county ...
1,E07000121,Lancaster,E05014894,Heysham North,North West,4801,Settled Working Families / Skilled Trades Suburbs,Post-Industrial Estates / Deprived Working Com...,0.474276,0.606460,2023.0,lab,independent,0.056387,0.776894,61.230889,99.418912,Review A,36.768382,84.495015,52.053803,100.0,True,No major caveat.,NaN,NaN
2,E07000125,Rossendale,E05015822,Haslingden,North West,7757,Settled Working Families / Skilled Trades Suburbs,Post-Industrial Estates / Deprived Working Com...,0.469640,0.684336,2024.0,lab,green,0.062996,0.946521,60.815359,99.194400,Review A,29.965193,85.670425,58.505654,100.0,True,No major caveat.,NaN,NaN
3,E06000007,Warrington,E05011038,Orford,North West,11962,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,0.382796,0.664762,2024.0,lab,independent,0.051640,0.900562,59.964809,98.534073,Review A,34.710333,81.012534,54.049729,100.0,True,No major caveat.,NaN,NaN
4,E07000125,Rossendale,E05015821,Hareholme & Waterfoot,North West,7128,Settled Working Families / Skilled Trades Suburbs,Post-Industrial Estates / Deprived Working Com...,0.457632,0.690274,2024.0,lab,green,0.066142,0.954112,59.559345,98.151083,Review A,29.278900,82.210659,58.594127,100.0,True,No major caveat.,NaN,NaN


## 19.8 Manual review exports

These files identify rows where manual review is advisable due to caveats or incomplete data.

In [8]:
manual_review_conditions = pd.Series(False, index=nw.index)

for col in ["target_model_ready"]:
    if col in nw.columns:
        manual_review_conditions |= ~nw[col].astype(str).str.lower().isin(["true", "1", "yes", "y"])

for col in ["boundary_caveat", "county_election_caveat"]:
    if col in nw.columns:
        manual_review_conditions |= nw[col].fillna("").ne("")

if "latest_election_allocated_valid_votes" in nw.columns:
    manual_review_conditions |= pd.to_numeric(nw["latest_election_allocated_valid_votes"], errors="coerce").fillna(0).le(0)

manual_review = nw[manual_review_conditions].copy()

manual_cols = [
    "LAD25CD", "LAD25NM", "WD25CD", "WD25NM",
    "initial_watchlist_score", "review_band",
    "target_model_ready", "data_confidence_note",
    "boundary_caveat", "county_election_caveat",
    "latest_election_source_year",
    "latest_election_allocated_valid_votes",
    "latest_election_margin_pct_allocated",
]
manual_cols = [c for c in manual_cols if c in manual_review.columns]

manual_review[manual_cols].to_csv(TABLE_DIR / "north_west_manual_review_rows_v1.csv", index=False)

print("Manual review rows:", len(manual_review))
display(manual_review[manual_cols].head(50))

Manual review rows: 154


,LAD25CD,LAD25NM,WD25CD,WD25NM,initial_watchlist_score,review_band,target_model_ready,data_confidence_note,boundary_caveat,county_election_caveat,latest_election_source_year,latest_election_allocated_valid_votes,latest_election_margin_pct_allocated
0,E07000117,Burnley,E05005152,Brunshaw,61.603515,Review A,True,County election caveat.,NaN,Latest election layer may include 2025 county ...,2025.0,1586.691161,0.074756
5,E07000121,Lancaster,E05014910,West End,58.914822,Review A,True,County election caveat.,NaN,Latest election layer may include 2025 county ...,2025.0,2916.000000,0.063443
9,E07000124,Ribble Valley,E05012013,Littlemoor,58.350736,Review A,True,County election caveat.,NaN,Latest election layer may include 2025 county ...,2025.0,901.875759,0.086640
17,E07000121,Lancaster,E05014901,Scale Hall,57.421809,Review A,True,County election caveat.,NaN,Latest election layer may include 2025 county ...,2025.0,2771.000000,0.118369
19,E07000121,Lancaster,E05014889,Carnforth & Millhead,57.243950,Review A,True,County election caveat.,NaN,Latest election layer may include 2025 county ...,2025.0,2252.754231,0.007566
20,E07000123,Preston,E05012202,Lea & Larches,57.054409,Review A,True,County election caveat.,NaN,Latest election layer may include 2025 county ...,2025.0,3551.000000,0.074908
22,E07000124,Ribble Valley,E05012010,Edisford & Low Moor,56.798769,Review A,True,County election caveat.,NaN,Latest election layer may include 2025 county ...,2025.0,1008.325747,0.086640
23,E07000117,Burnley,E05005163,Trinity,56.795593,Review A,True,County election caveat.,NaN,Latest election layer may include 2025 county ...,2025.0,1629.907626,0.192752
24,E07000124,Ribble Valley,E05012015,Primrose,56.450520,Review A,True,County election caveat.,NaN,Latest election layer may include 2025 county ...,2025.0,1057.197246,0.086640
29,E07000122,Pendle,E05015546,Barnoldswick,55.792409,Review A,True,County election caveat.,NaN,Latest election layer may include 2025 county ...,2025.0,2849.701395,0.053008


## 19.9 Sensitivity check

This creates two alternative score versions to see whether the same wards remain visible when weights change.

This is not a final judgement. It is a robustness check.

In [9]:
sensitivity = nw.copy()

weight_sets = {
    "baseline": {
        "demographic_relevance_score": 0.35,
        "electoral_opportunity_score": 0.30,
        "political_openness_score": 0.25,
        "data_confidence_score": 0.10,
    },
    "electoral_heavy": {
        "demographic_relevance_score": 0.25,
        "electoral_opportunity_score": 0.40,
        "political_openness_score": 0.25,
        "data_confidence_score": 0.10,
    },
    "demographic_heavy": {
        "demographic_relevance_score": 0.45,
        "electoral_opportunity_score": 0.25,
        "political_openness_score": 0.20,
        "data_confidence_score": 0.10,
    },
}

for name, weights in weight_sets.items():
    sensitivity[f"score_{name}"] = 0
    for component, weight in weights.items():
        if component in sensitivity.columns:
            sensitivity[f"score_{name}"] += pd.to_numeric(sensitivity[component], errors="coerce").fillna(0) * weight

    sensitivity[f"rank_{name}"] = sensitivity[f"score_{name}"].rank(ascending=False, method="min")

sensitivity["sensitivity_rank_range"] = sensitivity[
    [f"rank_{name}" for name in weight_sets]
].max(axis=1) - sensitivity[
    [f"rank_{name}" for name in weight_sets]
].min(axis=1)

sens_cols = [
    "LAD25CD", "LAD25NM", "WD25CD", "WD25NM",
    "dominant_cluster_name",
    "review_band",
    "score_baseline", "score_electoral_heavy", "score_demographic_heavy",
    "rank_baseline", "rank_electoral_heavy", "rank_demographic_heavy",
    "sensitivity_rank_range",
]

sens_cols = [c for c in sens_cols if c in sensitivity.columns]

sensitivity[sens_cols].sort_values("rank_baseline").to_csv(
    TABLE_DIR / "north_west_score_sensitivity_review_v1.csv",
    index=False
)

display(sensitivity[sens_cols].sort_values("rank_baseline").head(50))

,LAD25CD,LAD25NM,WD25CD,WD25NM,dominant_cluster_name,review_band,score_baseline,score_electoral_heavy,score_demographic_heavy,rank_baseline,rank_electoral_heavy,rank_demographic_heavy,sensitivity_rank_range
0,E07000117,Burnley,E05005152,Brunshaw,Post-Industrial Estates / Deprived Working Com...,Review A,61.603515,65.421639,58.622670,1.0,3.0,1.0,2.0
1,E07000121,Lancaster,E05014894,Heysham North,Settled Working Families / Skilled Trades Suburbs,Review A,61.230889,66.003552,58.080286,2.0,2.0,2.0,0.0
2,E07000125,Rossendale,E05015822,Haslingden,Settled Working Families / Skilled Trades Suburbs,Review A,60.815359,66.385882,56.603074,3.0,1.0,4.0,3.0
3,E06000007,Warrington,E05011038,Orford,Post-Industrial Estates / Deprived Working Com...,Review A,59.964809,64.595029,56.682729,4.0,6.0,3.0,3.0
4,E07000125,Rossendale,E05015821,Hareholme & Waterfoot,Settled Working Families / Skilled Trades Suburbs,Review A,59.559345,64.852520,55.446995,5.0,5.0,10.0,5.0
5,E07000121,Lancaster,E05014910,West End,Settled Working Families / Skilled Trades Suburbs,Review A,58.914822,62.562378,56.135518,6.0,16.0,5.0,11.0
6,E07000121,Lancaster,E05014885,Bare,Rooted Older Homeowners,Review A,58.745402,65.363806,53.656073,7.0,4.0,21.0,17.0
7,E07000119,Fylde,E05014539,Ashton,Rooted Older Homeowners,Review A,58.660754,64.406152,54.700804,8.0,7.0,13.0,6.0
8,E07000121,Lancaster,E05014900,Poulton,Post-Industrial Estates / Deprived Working Com...,Review A,58.393512,61.797488,55.766612,9.0,21.0,7.0,14.0
9,E07000124,Ribble Valley,E05012013,Littlemoor,Settled Working Families / Skilled Trades Suburbs,Review A,58.350736,63.838206,54.032170,10.0,8.0,18.0,10.0


## 19.10 Final checklist for the next modelling iteration

Before treating the watchlist as operationally meaningful, review:

1. Manual review rows
2. Sefton boundary caveats
3. Lancashire / county-electoral-division caveats
4. Score sensitivity
5. Whether component scores make sense geographically
6. Whether any council has obviously distorted results due to source-year differences

The next stage is either:
- refine scoring weights / caveats, or
- add organisational/candidate/member capacity data as a new component.